In [39]:
import os
import glob
import json
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

## Creating and Exploring the Dataset

The dataset is the data for a group of employees in a company InusreLLM created by Ed Donner. thisi sjust a test data set for now, i hope to use real world data soon. 
- 4 folders: Company, Contract, Employees and Products.
- All data is in Markdown format.

In [4]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
if openai_api_key:
    print(f"OpenAI API key exisits and starts with {openai_api_key[:5]}")
else:
    print("OpenAI API key does not exist")

NANO = "gpt-4.1-nano"

openai = OpenAI()



OpenAI API key exisits and starts with sk-pr


In [ ]:
knowledge = {}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    # Path() creates a path object, which is a wrapper around a string that represents a file path.
    # Path.stem returns the filename without the extension. i f the path is "knowledge-base/employees/John_Doe.txt"
    # , the stem is "John_Doe".
    # Path.split() splits the filename into a list of the names split into first name and last name.
    name =Path(filename).stem.split(" ")[-1]
    with open(filename, "r", encoding = "utf-8") as f:
        knowledge[name.lower()] = f.read()


In [29]:
knowledge.keys()
knowledge["chen"]


"# HR Record\n\n# Robert Chen\n\n## Summary\n- **Date of Birth:** February 28, 1983\n- **Job Title:** Senior Full Stack Engineer\n- **Location:** San Francisco, California\n- **Current Salary:** $152,000\n\n## Insurellm Career Progression\n- **January 2016 - Present:** Senior Full Stack Engineer\n  - Technical lead for Homellm home insurance portal\n  - Architects full-stack solutions using React, Node.js, and PostgreSQL\n  - Mentors team of 6 engineers\n  - Led 3 major platform releases\n\n- **June 2012 - December 2015:** Full Stack Developer at WebSolutions Inc.\n  - Built web applications for enterprise clients\n  - Worked with various JavaScript frameworks and backend technologies\n\n- **August 2008 - May 2012:** Software Engineer at TechStartup\n  - Developed features for SaaS platform\n  - Gained experience across full technology stack\n\n## Annual Performance History\n- **2023:** Rating: 4.8/5\n  *Exceptional performance. Led critical platform modernization project. Outstanding 

In [32]:
contracts = glob.glob("knowledge-base/contracts/*")

for contract in contracts:
    names = Path(contract).stem
    with open(contract, "r", encoding = "utf-8") as f:
        knowledge[names.lower()] = f.read()
print (knowledge.keys())

dict_keys(['chen', 'harper', 'thomson', 'foster', 'lancaster', 'walker', 'rodriguez', 'park', 'kim', 'carter', 'tran', 'wilson', 'adams', 'liu', 'blake', 'bishop', 'zhang', 'anderson', 'johnson', 'thompson', "o'brien", 'rivera', 'patel', 'spencer', 'sharma', 'martinez', 'greene', 'trenton', 'williams', 'brooks', 'contract with advantage medical coverage for healthllm', 'contract with apex reinsurance for rellm - ai-powered enterprise reinsurance solution', 'contract with atlantic risk solutions for bizllm', 'contract with belvedere insurance for markellm', 'contract with brightway solutions for markellm', 'contract with connectinsure agency for markellm', 'contract with continental commercial group for bizllm', 'contract with drivesmart insurance for carllm', 'contract with evergreen life insurance for lifellm', 'contract with everguard insurance for rellm - ai-powered enterprise reinsurance solution', 'contract with fasttrack insurance services for claimllm', 'contract with fortress b

In [ ]:
## Load all 76 documents, tag each with its folder name as doc_type
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    docs = loader.load()
    for doc in docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")


company
contracts
employees
products
Loaded 76 documents


In [ ]:
## Split documents and build one ChromaDB collection per doc_type

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

CHROMA_PATH = "chroma_db"
DOC_TYPES = ["employees", "contracts", "company", "products"]

vectorstores = {}
for doc_type in DOC_TYPES:
    type_docs = [d for d in documents if d.metadata.get("doc_type") == doc_type]
    chunks = splitter.split_documents(type_docs)
    vs = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=doc_type,
        persist_directory=CHROMA_PATH,
    )
    vectorstores[doc_type] = vs
    print(f"  {doc_type}: {len(chunks)} chunks indexed")

print("All collections built and persisted to", CHROMA_PATH)


In [ ]:
## Retrieval functions — each searches one ChromaDB collection and returns top-3 chunks

def _retrieve(doc_type: str, query: str, k: int = 3) -> str:
    retriever = vectorstores[doc_type].as_retriever(search_kwargs={"k": k})
    results = retriever.invoke(query)
    if not results:
        return f"No relevant information found in {doc_type}."
    return "\n\n---\n\n".join(
        f"[{doc_type.upper()} | source: {r.metadata.get('source', 'unknown')}]\n{r.page_content}"
        for r in results
    )

def search_employees(query: str) -> str:
    return _retrieve("employees", query)

def search_contracts(query: str) -> str:
    return _retrieve("contracts", query)

def search_company(query: str) -> str:
    return _retrieve("company", query)

def search_products(query: str) -> str:
    return _retrieve("products", query)

TOOL_FUNCTIONS = {
    "search_employees": search_employees,
    "search_contracts": search_contracts,
    "search_company": search_company,
    "search_products": search_products,
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "search_employees",
            "description": "Search InsureLLM employee HR records. Use for questions about staff, roles, salaries, performance, or career history.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query describing the information needed."}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_contracts",
            "description": "Search InsureLLM client contracts. Use for questions about agreements, pricing, SLAs, renewal dates, or partner names.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query describing the information needed."}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_company",
            "description": "Search InsureLLM company information. Use for questions about company overview, mission, structure, or policies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query describing the information needed."}
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search InsureLLM product documentation. Use for questions about product features, pricing tiers, or technical specs.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query describing the information needed."}
                },
                "required": ["query"],
            },
        },
    },
]

print("Tools defined:", [t["function"]["name"] for t in tools])


In [ ]:
## Agent loop — gpt-4.1-nano with iterative tool calling

SYSTEM_PROMPT = """You are a helpful assistant for InsureLLM, an AI-driven insurance technology company.
You have access to the company's internal knowledge base via four search tools:
- search_employees: HR records, performance reviews, salaries, career history
- search_contracts: Client contracts, SLAs, pricing, renewal terms
- search_company: Company overview, mission, structure, policies
- search_products: Product documentation, features, pricing tiers

Always use the appropriate tool(s) to retrieve relevant information before answering.
If a question spans multiple domains (e.g. an employee and their product), call multiple tools.
Base your answers strictly on the retrieved content. If information is not found, say so clearly."""


def run_agent(user_message: str, history: list) -> str:
    # Convert Gradio history (list of {"role": ..., "content": ...} dicts) to OpenAI messages
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history:
        if isinstance(turn, dict):
            messages.append({"role": turn["role"], "content": turn["content"]})
        else:
            # fallback for tuple-style history
            messages.append({"role": "user", "content": turn[0]})
            messages.append({"role": "assistant", "content": turn[1]})
    messages.append({"role": "user", "content": user_message})

    print(f"\n[Agent] User: {user_message}")

    max_iterations = 10
    for iteration in range(max_iterations):
        response = openai.chat.completions.create(
            model=NANO,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        choice = response.choices[0]
        messages.append(choice.message)

        if choice.finish_reason == "tool_calls":
            for tool_call in choice.message.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)
                print(f"  [Tool call] {fn_name}({fn_args})")

                if fn_name in TOOL_FUNCTIONS:
                    result = TOOL_FUNCTIONS[fn_name](**fn_args)
                else:
                    result = f"Unknown tool: {fn_name}"

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result,
                })
        else:
            print(f"  [Agent] Done after {iteration + 1} iteration(s)")
            return choice.message.content

    return "I was unable to complete the request within the allowed number of steps."


print("Agent loop ready.")


In [ ]:
## Gradio Chat UI

def chat(message: str, history: list) -> str:
    return run_agent(message, history)

demo = gr.ChatInterface(
    fn=chat,
    type="messages",
    title="InsureLLM Knowledge Assistant",
    description=(
        "Ask anything about InsureLLM employees, contracts, products, or company information. "
        "The agent will search the internal knowledge base and synthesise an answer."
    ),
    examples=[
        "What is Robert Chen's current salary and performance rating?",
        "Summarise the contract with Apex Reinsurance for Rellm.",
        "What products does InsureLLM offer?",
        "Who are the top-performing employees in 2023?",
    ],
)

demo.launch()
